# List 的插入代價與 Prefix Sum

1. `list.insert()` 為什麼慢：O(n)
2. 求和問題：從 O(n²) 暴力法到 O(n) 解法
3. Prefix Sum 前綴和的觀念與應用


## 1. List 缺點：插入是 O(n)

`list` 在中間或開頭插入元素時，後面所有元素都要搬家。


In [ ]:
from time import perf_counter

n = 50000

# 在開頭附近一直插入：很慢
start = perf_counter()
a = [0]
for i in range(1, n):
    a.insert(1, i)
print(f"insert 開頭: {perf_counter() - start:.4f} 秒")

# 在尾端 append：很快
start = perf_counter()
b = [0]
for i in range(1, n):
    b.append(i)
print(f"append 尾端: {perf_counter() - start:.4f} 秒")


如果需要頻繁在「開頭」插入或刪除，
應該用 `collections.deque`（見 `20_python_queue_stack_deque.ipynb`）
或 linked list 的思維，而不是 `list.insert()`。


## 2. 求和問題

【題目】給定 $n$ 個整數 $a_1, \dots, a_n$，求所有「兩兩相乘」的總和：

$$S = \sum_{i<j} a_i \times a_j$$

【範例輸入】

```text
5
1 2 3 4 5
```

【範例輸出】

```text
85
```


In [ ]:
sample = "5\n1 2 3 4 5"
data = sample.split("\n")


## 3. 解法 1：雙層迴圈暴力法 O(n²)

主要迴圈執行次數：$(n-1) + (n-2) + \dots + 1 = \dfrac{n(n-1)}{2}$

$n = 10^5$ 時約 $5 \times 10^9$ 次，Python 一定超時。


In [ ]:
def solve1(data):
    n = int(data[0])
    a = list(map(int, data[1].split()))
    s = 0
    for i in range(n):
        for j in range(i + 1, n):
            s += a[i] * a[j]
    return s

print(solve1(data))


## 4. Prefix Sum 前綴和

定義 $s_i = a_1 + a_2 + \dots + a_i$，就能把「區間和」變成 O(1) 查詢：

$$a_l + a_{l+1} + \dots + a_r = s_r - s_{l-1}$$

對這題來說：

$$S = \sum_{j} a_j \times (a_1 + \dots + a_{j-1}) = \sum_j a_j \times s_{j-1}$$


In [ ]:
def solve2(data):
    n = int(data[0])
    a = list(map(int, data[1].split()))
    prefix = 0       # 前 j-1 項的和
    ans = 0
    for j in range(n):
        ans += a[j] * prefix
        prefix += a[j]
    return ans

print(solve2(data))


## 5. 解法 3：數學公式 O(n)

$$\left(\sum a_i\right)^2 = \sum a_i^2 + 2\sum_{i<j} a_i a_j
\;\Rightarrow\;
S = \frac{(\sum a_i)^2 - \sum a_i^2}{2}$$


In [ ]:
def solve3(data):
    d = list(map(int, data[1].split()))
    s1 = sum([e ** 2 for e in d])
    s2 = sum(d) ** 2
    return (s2 - s1) // 2

print(solve3(data))


## 6. 三種解法速度比較


In [ ]:
import random
from time import perf_counter

n = 3000
big = [str(n), " ".join(str(random.randint(1, 100)) for _ in range(n))]

for f in [solve1, solve2, solve3]:
    start = perf_counter()
    r = f(big)
    print(f"{f.__name__}: 答案={r}, 時間={perf_counter() - start:.4f} 秒")


## 7. Prefix Sum 的標準寫法：區間和查詢

多次詢問「$a_l$ 到 $a_r$ 的和」時，先建表，每次查詢 O(1)。


In [ ]:
a = [2, 1, 4, 7, 4, 8, 3, 6, 4, 7]
n_ = len(a)

# prefix[i] = 前 i 個元素的和，prefix[0] = 0
prefix = [0] * (n_ + 1)
for i in range(n_):
    prefix[i + 1] = prefix[i] + a[i]
print(prefix)

def range_sum(l, r):        # a[l..r]，0-based，含兩端
    return prefix[r + 1] - prefix[l]

print(range_sum(2, 5))      # 4+7+4+8 = 23
print(range_sum(0, 9))      # 全部 = 46


## 8. 練習題

給定 `a = [3, -1, 4, -1, 5, -9, 2, 6]`，
用前綴和回答：`a[1..4]` 與 `a[3..7]` 的區間和。

期望輸出：

```text
7
3
```


In [ ]:
# 練習作答區
a = [3, -1, 4, -1, 5, -9, 2, 6]


### 參考答案

In [ ]:
a = [3, -1, 4, -1, 5, -9, 2, 6]
prefix = [0]
for x in a:
    prefix.append(prefix[-1] + x)
print(prefix[5] - prefix[1])
print(prefix[8] - prefix[3])
